# Creating the `LIB` table

From the `books_df` that was created in a different notebook I will contruct a `LIB` table.

## Import Packages

In [154]:
import pandas as pd
import re

## Set the Data Path

In [176]:
file_location = 'C:/Users/mason/Box/Text As Data Final/Output'
books_df = pd.read_csv(file_location + '/BrandonSandersonBooks.csv')
books_df.head()

,title,file_path,author,date,chapter_id,paragraph_id,text
0,A Memory of Light,C:/Users/mason/Box/Text As Data Final/Sanderso...,Robert Jordan & Brandon Sanderson,2013-01-08,0,0,by Robert Jordan
1,A Memory of Light,C:/Users/mason/Box/Text As Data Final/Sanderso...,Robert Jordan & Brandon Sanderson,2013-01-08,0,1,The Eye of the World
2,A Memory of Light,C:/Users/mason/Box/Text As Data Final/Sanderso...,Robert Jordan & Brandon Sanderson,2013-01-08,0,2,The Great Hunt
3,A Memory of Light,C:/Users/mason/Box/Text As Data Final/Sanderso...,Robert Jordan & Brandon Sanderson,2013-01-08,0,3,The Dragon Reborn
4,A Memory of Light,C:/Users/mason/Box/Text As Data Final/Sanderso...,Robert Jordan & Brandon Sanderson,2013-01-08,0,4,The Shadow Rising


In [177]:
Elantris_df = books_df[books_df['title'] == 'Elantris']
with pd.option_context('display.max_rows', 150, 'display.max_colwidth', 100):
    print(Elantris_df['text'].head(150))

16914                                                                                       BRANDONSANDERSON
16915                                                         A TOM DOHERTY ASSOCIATES BOOK NEW YORKNew York
16916    This is a work of fiction. All the characters and events portrayed in this novel are either fict...
16917                                                                                               ELANTRIS
16918                                                                  Copyright © 2005 by Brandon Sanderson
16919     All rights reserved, including the right to reproduce this book, or portions thereof, in any form.
16920                                                               This book is printed on acid-free paper.
16921                                                  Spot illustrations throughout by Stephen de las Heras
16922                                                                      Book design by Milenda Nan Ok Lee
16923              

## Correct the chapter_id

I don't want all of the beginning cruft but I do want the prologue for each book as chapter 0

In [178]:
start_pattern = r'^[\W_]*(?:Prologue|Preface|Introduction|Foreword|Chapter 1|Chapter One|Book One|Book 1|Part One|Part 1|The Wheel of Time turns|GOD.S DEATH|YOU WANT|Eshonai|Yeah. he.dheard|Nomad woke up|In the middle|Right about then)\b'
end_pattern = r'^\s*(?:Acknowledgments|Endnotes|About the Author|Glossary|Ars Arcanum|Postscript|POSTSCRIPT|Epilogue|EPILOGUE|Afterword|Also by Brandon Sanderson|Copyright|Robert Jordan & Brandon Sanderson|Illustrations:|THE END)\b'

books_df['temp_word_count'] = books_df['text'].str.split().str.len()
books_df['is_short_line'] = books_df['temp_word_count'] < 10
books_df['short_density'] = books_df.groupby('file_path')['is_short_line'].transform(
    lambda x: x.rolling(window=15, center=True, min_periods=1).sum()
)
is_not_toc = books_df['short_density'] < 12

raw_is_start = books_df['text'].str.contains(start_pattern, flags=re.IGNORECASE, regex=True, na=False)

books_df['is_start'] = raw_is_start & is_not_toc
books_df['has_started'] = books_df.groupby('file_path')['is_start'].cumsum()

book_has_any_start = books_df.groupby('file_path')['is_start'].transform('max')

valid_start_mask = (books_df['has_started'] > 0) | (book_has_any_start == 0) & (books_df['chapter_id']> 0)

raw_is_end = books_df['text'].str.contains(end_pattern, flags=re.IGNORECASE, regex=True, na=False)
books_df['is_end'] = raw_is_end & valid_start_mask & is_not_toc
books_df['has_ended'] = books_df.groupby('file_path')['is_end'].cumsum()

valid_end_mask = (books_df['has_ended'] == 0)

clean_books_df = books_df[valid_start_mask & valid_end_mask].copy()

clean_books_df = clean_books_df.drop(columns=['is_start', 'is_end', 'has_started', 'has_ended', 'is_short_line', 'short_density', 'temp_word_count'])
clean_books_df = clean_books_df.reset_index(drop=True)

print(f"Original number of rows: {len(books_df)}")
print(f"Cleaned number of rows: {len(clean_books_df)}")

LIB_titles = clean_books_df['title'].unique()
print(f"Unique book titles in the cleaned dataset ({len(LIB_titles)}):")
print(LIB_titles)

Original number of rows: 287646
Cleaned number of rows: 77473
Unique book titles in the cleaned dataset (24):
<StringArray>
[             'A Memory of Light',              'Arcanum Unbounded',
                       'Elantris',         'Isles of the Emberdark',
                    'Oathbringer',                  'Rhythm of War',
                'Shadows of Self',                     'Steelheart',
            'The Aether of Night',               'The Alloy of Law',
          'The Bands of Mourning',               'The Final Empire',
               'The Hero of Ages',                 'The Lost Metal',
                 'The Sunlit Man',               'The Way of Kings',
          'The Well of Ascension',              'The Wheel of Time',
             'Towers of Midnight',       'Tress of the Emerald Sea',
                     'Warbreaker',                 'Wind and Truth',
              'Words of Radiance', 'Yumi and the Nightmare Painter']
Length: 24, dtype: str


## Still getting wierd Chapter Patterns lets clean that up

In [179]:
chapter_pattern = r'^\s*(?:Prologue|Preface|Introduction|Foreword|Chapter \d+|Chapter [A-Za-z]+|Book \d+|Book [A-Za-z]+|Part \d+|Part [A-Za-z]+)\b'
clean_books_df['is_new_chapter'] = clean_books_df['text'].str.contains(
    chapter_pattern, flags=re.IGNORECASE, regex=True, na=False)
clean_books_df['logical_chapter_id'] = clean_books_df.groupby('file_path')['is_new_chapter'].cumsum()
clean_books_df = clean_books_df.drop(columns=['is_new_chapter'])

## Diagnostic View of each Title

In [180]:
diagnostic_df = clean_books_df.copy()
diagnostic_df['text_preview'] = diagnostic_df['text'].astype(str).str[:100].str.replace('\n', ' ')+'...'


first_few = diagnostic_df.groupby('title').head(5)
last_few = diagnostic_df.groupby('title').tail(5)


boundaries_df = pd.concat([first_few, last_few]).drop_duplicates().sort_values(['title', 'logical_chapter_id'])

In [181]:
view_df = boundaries_df[['title', 'logical_chapter_id', 'text_preview']]

diagnostic_path = file_location + '/diagnostic_boundaries.csv'
view_df.to_csv(diagnostic_path, index=False)

## Get Book Length, number of chapters, Number of Paragraphs

In [182]:
clean_books_df['word_count'] = clean_books_df['text'].str.split().str.len()
lib_df = clean_books_df.groupby(
    ['title', 'file_path', 'author', 'date'],
    dropna=False
).agg(
    length_in_words=('word_count', 'sum'),
    total_chapters=('logical_chapter_id', 'nunique'),
    total_paragraphs=('text', 'count')
).reset_index()
lib_df = lib_df.rename(columns={
    'file_path': 'File_path',
    'author': 'Author',
    'date': 'Date',
    'length_in_words': 'Length',
    'total_chapters': 'Total_chapters',
    'total_paragraphs': 'Total_paragraphs'
})

lib_df = lib_df.sort_values('title').reset_index(drop=True)

clean_books_df = clean_books_df.drop(columns=['word_count'])

with pd.option_context('display.max_rows', None, 'display.max_columns', 1000):
    print(lib_df)

                             title  \
0                A Memory of Light   
1                Arcanum Unbounded   
2                         Elantris   
3           Isles of the Emberdark   
4                      Oathbringer   
5                    Rhythm of War   
6                  Shadows of Self   
7                       Steelheart   
8              The Aether of Night   
9                 The Alloy of Law   
10           The Bands of Mourning   
11                The Final Empire   
12                The Hero of Ages   
13                  The Lost Metal   
14                  The Sunlit Man   
15                The Way of Kings   
16           The Well of Ascension   
17               The Wheel of Time   
18              Towers of Midnight   
19        Tress of the Emerald Sea   
20                      Warbreaker   
21                  Wind and Truth   
22               Words of Radiance   
23  Yumi and the Nightmare Painter   

                                            File_